<center>
  <a>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" style="zoom: 50%;">
    <div style="height: 20px;"></div> <!-- Empty div for spacing -->
    <h1 </h1>
  </a>
</center>

#RAG Evaluation with Evidently

This notebook demonstrates a Retrieval-Augmented Generation (RAG) system evaluation using FAISS for efficient context retrieval and Evidently AI for assessing retrieval and generation quality. A synthetic dataset of questions, contexts, and responses is used, with contexts embedded via a sentence transformer and retrieved using FAISS.

**Installation and Imports**

In [1]:
# Install Evidently if not already installed
!pip install evidently openai langchain --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.0/238.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.3/568.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.9 MB/s eta 0:00:00


In [2]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from evidently import Dataset, DataDefinition
from evidently.descriptors import *
from evidently.presets import TextEvals
from evidently.tests import *
from evidently.ui.workspace import CloudWorkspace
from IPython.display import display, HTML
import os

**Set OpenAI API Key**

In [ ]:
os.environ["OPENAI_API_KEY"] = "your key"  # Replace with your OpenAI API key

**Setting Up the Retrieval System with FAISS**

In [11]:
# Synthetic corpus of contexts
corpus = [
    "The capital of Japan is Tokyo, known for its vibrant culture.",
    "Leaves change color in autumn due to reduced chlorophyll levels.",
    "Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km.",
    "Airplanes stay aloft due to lift generated by their wings and thrust from engines.",
    "The sky appears blue because of Rayleigh scattering of sunlight.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",
    "Penicillin was discovered by Alexander Fleming in 1928.",
    "Earthquakes are caused by the movement of tectonic plates.",
    "Greenhouse gases like CO2 trap heat, causing global warming.",
    "Vaccines stimulate the immune system to recognize and fight pathogens.",
    "Stars twinkle due to atmospheric turbulence distorting their light.",
    "Blockchain is a decentralized ledger system secured by cryptography.",
]

# Load sentence transformer model for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the corpus
corpus_embeddings = embedder.encode(corpus, convert_to_numpy=True)

# Create a FAISS index (FlatL2 for simplicity)
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)

# Verify the index
print(f"FAISS index contains {index.ntotal} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index contains 12 vectors


 **Retrieving Contexts with FAISS**

In [12]:
# Synthetic dataset of questions and generated responses
synthetic_data = [
    ["What is the capital of Japan?", "Tokyo is the capital."],
    ["Why do leaves change color in autumn?", "Because of chemical changes."],
    ["What is the largest planet?", "Jupiter is the biggest."],
    ["How do airplanes stay in the air?", "Due to lift and thrust."],
    ["Why is the sky blue?", "Due to light scattering."],
    ["What is photosynthesis?", "Plants make energy from sunlight."],
    ["Who discovered penicillin?", "Alexander Fleming found it."],
    ["What causes earthquakes?", "Due to plate tectonics."],
    ["What causes global warming?", "Due to greenhouse gases."],
    ["How do vaccines work?", "They train the immune system."],
]
columns = ["Question", "Response"]
data_df = pd.DataFrame(synthetic_data, columns=columns)

# Retrieve top-2 contexts for each question using FAISS
def retrieve_contexts(question, embedder, index, corpus, k=2):
    question_embedding = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(question_embedding, k)
    return [corpus[idx] for idx in indices[0]]

# Add retrieved contexts to the dataframe
data_df['Context'] = data_df['Question'].apply(lambda q: retrieve_contexts(q, embedder, index, corpus, k=2))

In [13]:
pd.set_option('display.max_colwidth', None)
data_df.head()

,Question,Response,Context
0,What is the capital of Japan?,Tokyo is the capital.,"[The capital of Japan is Tokyo, known for its vibrant culture., Blockchain is a decentralized ledger system secured by cryptography.]"
1,Why do leaves change color in autumn?,Because of chemical changes.,"[Leaves change color in autumn due to reduced chlorophyll levels., Photosynthesis is the process by which plants convert sunlight into energy.]"
2,What is the largest planet?,Jupiter is the biggest.,"[Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km., The sky appears blue because of Rayleigh scattering of sunlight.]"
3,How do airplanes stay in the air?,Due to lift and thrust.,"[Airplanes stay aloft due to lift generated by their wings and thrust from engines., Stars twinkle due to atmospheric turbulence distorting their light.]"
4,Why is the sky blue?,Due to light scattering.,"[The sky appears blue because of Rayleigh scattering of sunlight., Stars twinkle due to atmospheric turbulence distorting their light.]"


**Evaluating Retrieval with Evidently**

In [14]:
context_eval = Dataset.from_pandas(
    data_df,
    data_definition=DataDefinition(text_columns=["Question", "Context", "Response"]),
    descriptors=[
        ContextRelevance("Question", "Context", output_scores=True, aggregation_method="mean", method="llm", alias="Mean Relevance"),
        ContextRelevance("Question", "Context", output_scores=True, aggregation_method="hit", method="llm", alias="Hit Rate")
    ]
)
context_eval.as_dataframe()

,Question,Response,Context,Mean Relevance,Mean Relevance scores,Hit Rate,Hit Rate scores
0,What is the capital of Japan?,Tokyo is the capital.,"[The capital of Japan is Tokyo, known for its vibrant culture., Blockchain is a decentralized ledger system secured by cryptography.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
1,Why do leaves change color in autumn?,Because of chemical changes.,"[Leaves change color in autumn due to reduced chlorophyll levels., Photosynthesis is the process by which plants convert sunlight into energy.]",0.55,"[1.0, 0.1]",1,"[1.0, 0.0]"
2,What is the largest planet?,Jupiter is the biggest.,"[Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km., The sky appears blue because of Rayleigh scattering of sunlight.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
3,How do airplanes stay in the air?,Due to lift and thrust.,"[Airplanes stay aloft due to lift generated by their wings and thrust from engines., Stars twinkle due to atmospheric turbulence distorting their light.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
4,Why is the sky blue?,Due to light scattering.,"[The sky appears blue because of Rayleigh scattering of sunlight., Stars twinkle due to atmospheric turbulence distorting their light.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
5,What is photosynthesis?,Plants make energy from sunlight.,"[Photosynthesis is the process by which plants convert sunlight into energy., Leaves change color in autumn due to reduced chlorophyll levels.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
6,Who discovered penicillin?,Alexander Fleming found it.,"[Penicillin was discovered by Alexander Fleming in 1928., Vaccines stimulate the immune system to recognize and fight pathogens.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
7,What causes earthquakes?,Due to plate tectonics.,"[Earthquakes are caused by the movement of tectonic plates., Greenhouse gases like CO2 trap heat, causing global warming.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
8,What causes global warming?,Due to greenhouse gases.,"[Greenhouse gases like CO2 trap heat, causing global warming., Earthquakes are caused by the movement of tectonic plates.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"
9,How do vaccines work?,They train the immune system.,"[Vaccines stimulate the immune system to recognize and fight pathogens., Blockchain is a decentralized ledger system secured by cryptography.]",0.50,"[1.0, 0.0]",1,"[1.0, 0.0]"


**Evaluating Generation with Evidently**

In [15]:
faithfulness_eval = Dataset.from_pandas(
    data_df,
    data_definition=DataDefinition(text_columns=["Question", "Context", "Response"]),
    descriptors=[
        FaithfulnessLLMEval("Response", context="Context", alias="Faithfulness"),
        TextLength("Response", alias="Response Length")
    ]
)
faithfulness_eval.as_dataframe()

,Question,Response,Context,Faithfulness,Faithfulness reasoning,Response Length
0,What is the capital of Japan?,Tokyo is the capital.,"[The capital of Japan is Tokyo, known for its vibrant culture., Blockchain is a decentralized ledger system secured by cryptography.]",FAITHFUL,"The response accurately states that Tokyo is the capital of Japan, which is confirmed by the source information.",21
1,Why do leaves change color in autumn?,Because of chemical changes.,"[Leaves change color in autumn due to reduced chlorophyll levels., Photosynthesis is the process by which plants convert sunlight into energy.]",UNFAITHFUL,"The response 'Because of chemical changes' does not accurately use the information from the source, which specifically states that leaves change color due to reduced chlorophyll levels. It also adds new information without grounding it in the provided source.",28
2,What is the largest planet?,Jupiter is the biggest.,"[Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km., The sky appears blue because of Rayleigh scattering of sunlight.]",FAITHFUL,"The statement 'Jupiter is the biggest' is a simplified assertion that aligns with the information provided in the source, which states that Jupiter is the largest planet in our solar system. While the response does not include the specific diameter, it is still accurate in its essence.",23
3,How do airplanes stay in the air?,Due to lift and thrust.,"[Airplanes stay aloft due to lift generated by their wings and thrust from engines., Stars twinkle due to atmospheric turbulence distorting their light.]",FAITHFUL,"The response mentions 'lift' and 'thrust', which are directly referenced in the source regarding how airplanes stay aloft. While it does not provide the full context, it accurately captures key terms from the source.",23
4,Why is the sky blue?,Due to light scattering.,"[The sky appears blue because of Rayleigh scattering of sunlight., Stars twinkle due to atmospheric turbulence distorting their light.]",UNFAITHFUL,"The response 'Due to light scattering' is vague and does not specify that it refers to the blue appearance of the sky due to Rayleigh scattering, which is provided in the SOURCE. Therefore, it doesn't accurately use the information from the SOURCE and is considered unfaithful.",24
5,What is photosynthesis?,Plants make energy from sunlight.,"[Photosynthesis is the process by which plants convert sunlight into energy., Leaves change color in autumn due to reduced chlorophyll levels.]",FAITHFUL,"The text accurately conveys the concept that plants convert sunlight into energy, reflecting the information in the source about photosynthesis.",33
6,Who discovered penicillin?,Alexander Fleming found it.,"[Penicillin was discovered by Alexander Fleming in 1928., Vaccines stimulate the immune system to recognize and fight pathogens.]",FAITHFUL,"The phrase 'Alexander Fleming found it' is a paraphrase of the original information in the source stating that 'Penicillin was discovered by Alexander Fleming in 1928.' This response maintains the core information regarding the discovery of penicillin and correctly attributes it to Alexander Fleming, thus it is faithful to the source.",27
7,What causes earthquakes?,Due to plate tectonics.,"[Earthquakes are caused by the movement of tectonic plates., Greenhouse gases like CO2 trap heat, causing global warming.]",FAITHFUL,"The response 'Due to plate tectonics' is a partial, accurate explanation that aligns with the information from the source stating that 'Earthquakes are caused by the movement of tectonic plates.'",23
8,What causes global warming?,Due to greenhouse gases.,"[Greenhouse gases like CO2 trap heat, causing global warming., Earthquakes are caused by the movement of tectonic plates.]",FAITHFUL,The text 'Due to greenhouse gases.' is faithful as it refers to the information in the source regarding greenhouse gases trapping heat and causing global warming. It accurately reflects the top

**Adding Ground Truth for Correctness**

In [16]:
data_df['Target'] = [
    "The capital is Tokyo.",
    "Due to reduced chlorophyll.",
    "Jupiter is the largest planet.",
    "Lift from wings keeps planes aloft.",
    "Rayleigh scattering causes blue skies.",
    "Plants convert sunlight to energy.",
    "Alexander Fleming discovered penicillin.",
    "Tectonic plate movements cause earthquakes.",
    "Greenhouse gases trap heat.",
    "Vaccines stimulate immunity."
]

In [19]:
correctness_eval = Dataset.from_pandas(
    data_df,
    data_definition=DataDefinition(text_columns=["Question", "Context", "Response", "Target"]),
    descriptors=[
        CorrectnessLLMEval("Response", target_output="Target", alias="Correctness"),
        BERTScore(columns=["Response", "Target"], alias="BERTScore"),

    ]
)
correctness_eval.as_dataframe()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Question,Response,Context,Target,Correctness,Correctness reasoning,BERTScore
0,What is the capital of Japan?,Tokyo is the capital.,"[The capital of Japan is Tokyo, known for its vibrant culture., Blockchain is a decentralized ledger system secured by cryptography.]",The capital is Tokyo.,CORRECT,"The OUTPUT conveys the same fact as the REFERENCE, simply rephrasing 'The capital is Tokyo' to 'Tokyo is the capital.' Both statements communicate the same information without any inaccuracies or omissions.",0.880078
1,Why do leaves change color in autumn?,Because of chemical changes.,"[Leaves change color in autumn due to reduced chlorophyll levels., Photosynthesis is the process by which plants convert sunlight into energy.]",Due to reduced chlorophyll.,INCORRECT,The OUTPUT introduces a different cause ('chemical changes') that does not correspond to the cause in the REFERENCE ('reduced chlorophyll'). This changes the original meaning and does not convey the same facts.,0.629697
2,What is the largest planet?,Jupiter is the biggest.,"[Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km., The sky appears blue because of Rayleigh scattering of sunlight.]",Jupiter is the largest planet.,CORRECT,"The OUTPUT states that 'Jupiter is the biggest', which conveys the same fact that Jupiter is the largest planet, albeit using a different word ('biggest' instead of 'largest'). There are no inaccuracies or omissions that change the original meaning.",0.846640
3,How do airplanes stay in the air?,Due to lift and thrust.,"[Airplanes stay aloft due to lift generated by their wings and thrust from engines., Stars twinkle due to atmospheric turbulence distorting their light.]",Lift from wings keeps planes aloft.,INCORRECT,"The OUTPUT mentions 'lift and thrust,' which introduces additional information not present in the REFERENCE and does not convey the same fact about lift specifically keeping planes aloft. Therefore, it contradicts the original meaning of the REFERENCE.",0.658462
4,Why is the sky blue?,Due to light scattering.,"[The sky appears blue because of Rayleigh scattering of sunlight., Stars twinkle due to atmospheric turbulence distorting their light.]",Rayleigh scattering causes blue skies.,INCORRECT,The output 'Due to light scattering' does not convey the specific fact from the reference that 'Rayleigh scattering causes blue skies.' It omits the critical detail of Rayleigh scattering and therefore incorrectly represents the information.,0.769216
5,What is photosynthesis?,Plants make energy from sunlight.,"[Photosynthesis is the process by which plants convert sunlight into energy., Leaves change color in autumn due to reduced chlorophyll levels.]",Plants convert sunlight to energy.,CORRECT,"The OUTPUT conveys the same facts as the REFERENCE, stating that plants convert sunlight into energy, just using slightly different wording. There are no contradictions or omissions in meaning.",0.885450
6,Who discovered penicillin?,Alexander Fleming found it.,"[Penicillin was discovered by Alexander Fleming in 1928., Vaccines stimulate the immune system to recognize and fight pathogens.]",Alexander Fleming discovered penicillin.,CORRECT,"The OUTPUT, 'Alexander Fleming found it.', conveys the same fact as the REFERENCE that Alexander Fleming discovered penicillin. Although the wording is slightly different, it maintains the original meaning without introducing inaccuracies or omissions.",0.677686
7,What causes earthquakes?,Due to plate tectonics.,"[Earthquakes are caused by the movement of tectonic plates., Greenhouse gases like CO2 trap heat, causing global warming.]",Tectonic plate movements cause earthquakes.,INCORRECT,The output 'Due to plate tectonics' does not explicitly convey the fact that tectonic plate movements cause earthquakes. It is unclear and incomplete as it does not state the cause-and-effect relationship explicitly.,0.778334
8,What causes global warming?,Due to greenhouse gases.,"[Greenhouse gases l

**Additional checks on response tone and refusal behavior.**

In [20]:
tone_eval = Dataset.from_pandas(
    data_df,
    data_definition=DataDefinition(text_columns=["Question", "Context", "Response"]),
    descriptors=[
        Sentiment("Response", alias="Response Sentiment"),
        DeclineLLMEval("Response", alias="Refusal Check")
    ]
)
tone_eval.as_dataframe()

,Question,Response,Context,Target,Response Sentiment,Refusal Check,Refusal Check reasoning
0,What is the capital of Japan?,Tokyo is the capital.,"[The capital of Japan is Tokyo, known for its vibrant culture., Blockchain is a decentralized ledger system secured by cryptography.]",The capital is Tokyo.,0.0000,OK,"The text provided is a factual statement about Tokyo being the capital, which does not involve a refusal or rejection of any sort. Therefore, it fits the OK category."
1,Why do leaves change color in autumn?,Because of chemical changes.,"[Leaves change color in autumn due to reduced chlorophyll levels., Photosynthesis is the process by which plants convert sunlight into energy.]",Due to reduced chlorophyll.,0.0000,UNKNOWN,The text 'Because of chemical changes.' does not provide enough context to determine if it is a decline or an acceptable response. It is unclear what request or situation it is addressing.
2,What is the largest planet?,Jupiter is the biggest.,"[Jupiter is the largest planet in our solar system, with a diameter of about 139,820 km., The sky appears blue because of Rayleigh scattering of sunlight.]",Jupiter is the largest planet.,0.0000,OK,"The statement 'Jupiter is the biggest' is an informative declaration and does not indicate any refusal or rejection. Therefore, it falls under the category of OK."
3,How do airplanes stay in the air?,Due to lift and thrust.,"[Airplanes stay aloft due to lift generated by their wings and thrust from engines., Stars twinkle due to atmospheric turbulence distorting their light.]",Lift from wings keeps planes aloft.,0.0000,UNKNOWN,The text 'Due to lift and thrust.' does not provide enough context to determine if it is a decline or if it is acceptable. It lacks a clear indication of refusal or a statement of agreement.
4,Why is the sky blue?,Due to light scattering.,"[The sky appears blue because of Rayleigh scattering of sunlight., Stars twinkle due to atmospheric turbulence distorting their light.]",Rayleigh scattering causes blue skies.,0.0000,UNKNOWN,The text 'Due to light scattering.' does not clearly indicate whether it is a refusal or a polite rejection. It lacks context to determine if it is a decline or simply a neutral explanation.
5,What is photosynthesis?,Plants make energy from sunlight.,"[Photosynthesis is the process by which plants convert sunlight into energy., Leaves change color in autumn due to reduced chlorophyll levels.]",Plants convert sunlight to energy.,0.2732,OK,"The text provides information about how plants make energy from sunlight, which is a neutral statement and does not involve any refusal or rejection."
6,Who discovered penicillin?,Alexander Fleming found it.,"[Penicillin was discovered by Alexander Fleming in 1928., Vaccines stimulate the immune system to recognize and fight pathogens.]",Alexander Fleming discovered penicillin.,0.0000,OK,"The statement presents a factual piece of information about Alexander Fleming, which does not involve a refusal or rejection."
7,What causes earthquakes?,Due to plate tectonics.,"[Earthquakes are caused by the movement of tectonic plates., Greenhouse gases like CO2 trap heat, causing global warming.]",Tectonic plate movements cause earthquakes.,0.0000,UNKNOWN,"The provided text 'Due to plate tectonics.' does not contain any clear indication of a refusal or rejection. It seems to be an incomplete statement that lacks context, making it difficult to classify it as a decline or an okay response."
8,What causes global warming?,Due to greenhouse gases.,"[Greenhouse gases like CO2 trap heat, causing global warming., Earthquakes are caused by the movement of tectonic plates.]",Greenhouse gases trap heat.,0.0000,UNKNOWN,"The provided text 'Due to greenhouse gases.' does not include an explicit refusal or rejection, nor does it provide sufficient context to determine if it is related to a decline. It appears to be a fragment or a response needing more information to classify it clearly."
9,How do vaccines work?,

**Displaying Evaluation Results**

In [21]:
from IPython.display import display, HTML

combined_eval = Dataset.from_pandas(
    data_df,
    data_definition=DataDefinition(text_columns=["Question", "Context", "Response", "Target"]),
    descriptors=[
        ContextRelevance("Question", "Context", output_scores=True, aggregation_method="mean", method="llm", alias="Mean Relevance"),
        FaithfulnessLLMEval("Response", context="Context", alias="Faithfulness"),
        CorrectnessLLMEval("Response", target_output="Target", alias="Correctness")
    ]
)

# Get evaluation results as dataframe
results_df = combined_eval.as_dataframe()

# Create HTML table
html_content = """
<h2>RAG Evaluation Results</h2>
<table border='1'>
    <tr>
        <th>Question</th>
        <th>Mean Relevance</th>
        <th>Faithfulness</th>
        <th>Correctness</th>
    </tr>
"""
for _, row in results_df.iterrows():
    html_content += f"""
    <tr>
        <td>{row['Question']}</td>
        <td>{row.get('Mean Relevance', 'N/A'):.2f}</td>
        <td>{row.get('Faithfulness', 'N/A')}</td>
        <td>{row.get('Correctness', 'N/A')}</td>
    </tr>
    """
html_content += "</table>"

# Display in Colab
display(HTML(html_content))

# Save as HTML file for fallback
with open("rag_evaluation_results.html", "w", encoding="utf-8") as f:
    f.write(html_content)
print("Results saved as 'rag_evaluation_results.html'. Download and open in a browser if display fails.")

Question,Mean Relevance,Faithfulness,Correctness
What is the capital of Japan?,0.50,FAITHFUL,CORRECT
Why do leaves change color in autumn?,0.50,UNFAITHFUL,INCORRECT
What is the largest planet?,0.50,FAITHFUL,CORRECT
How do airplanes stay in the air?,0.50,FAITHFUL,INCORRECT
Why is the sky blue?,0.00,FAITHFUL,INCORRECT
What is photosynthesis?,0.50,FAITHFUL,CORRECT
Who discovered penicillin?,0.50,FAITHFUL,CORRECT
What causes earthquakes?,0.50,FAITHFUL,INCORRECT
What causes global warming?,0.00,FAITHFUL,INCORRECT
How do vaccines work?,0.50,FAITHFUL,CORRECT


Results saved as 'rag_evaluation_results.html'. Download and open in a browser if display fails.
